# InsightVLM — NB4: Qwen2-VL-2B + QLoRA

The generative model: produces the finding, the reasoning chain and the insight,
not just a class id. **Completely independent of NB1–NB3** — run it on a separate
account if GPU quota is tight. ~2.3 h. Output: `nb4_results.json`

**After the install cell, restart the kernel**, then run the rest. `transformers`
is preinstalled on Kaggle; upgrading it in-session leaves the old module loaded
and `Qwen2VLForConditionalGeneration` may not resolve until a restart.

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate peft trl bitsandbytes qwen-vl-utils
print("INSTALL DONE -- now RESTART THE KERNEL, then run the cells below.")

In [ ]:
import os, sys, json, time, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset as TorchDS

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
DATA = DATA_ROOT if (DATA_ROOT / "records.jsonl").exists() else \
       list(Path("/kaggle/input").glob("**/records.jsonl"))[0].parent
print("dataset:", DATA)
meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
te = meta[meta.split == "test"].reset_index(drop=True)
_ci = json.loads((DATA / "class_index.json").read_text())["label_to_id"]
CLASSES = sorted(_ci, key=_ci.get)
print(f"{len(meta):,} samples | {len(CLASSES)} classes | test {len(te):,}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
from transformers import (AutoProcessor, Qwen2VLForConditionalGeneration,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
# 1500 keeps the run near 1.5 h on a T4. Raise it if quota allows -- the
# size is printed and stored in nb4_results.json, so report whatever you use.
N_TRAIN = 1500
rows = [json.loads(l) for l in open(DATA / "instruct" / "train.jsonl")][:N_TRAIN]
for r in rows: r["image"] = str(DATA / r["image"])
print(len(rows), "training samples (subset -- report this size in the paper)")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)   # T4 is fp16, not bf16
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
vlm = prepare_model_for_kbit_training(vlm); vlm.config.use_cache = False
vlm = get_peft_model(vlm, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
vlm.print_trainable_parameters()

In [ ]:
class VLMDataset(TorchDS):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    texts, images = [], []
    for b in batch:
        msgs = [{"role":"user","content":[{"type":"image","image":b["image"]},
                 {"type":"text","text":b["messages"][0]["content"].replace("<image>\n","")}]},
                {"role":"assistant","content":[{"type":"text","text":b["messages"][1]["content"]}]}]
        texts.append(processor.apply_chat_template(msgs, tokenize=False))
        imgs, _ = process_vision_info(msgs); images.append(imgs)
    enc = processor(text=texts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=1024)
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    for tok in ("<|image_pad|>","<|vision_start|>","<|vision_end|>"):
        tid = processor.tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid >= 0: labels[labels == tid] = -100
    enc["labels"] = labels
    return enc

# TrainingArguments' signature changes between transformers releases -- on the
# Kaggle image `warmup_ratio` raised TypeError. Rather than pin a version, keep
# only the kwargs this build actually accepts and report what was dropped.
import inspect
_want = dict(output_dir="insightvlm-lora", num_train_epochs=1,
             per_device_train_batch_size=1, gradient_accumulation_steps=8,
             learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
             logging_steps=10, save_strategy="epoch", fp16=True,
             gradient_checkpointing=True, optim="paged_adamw_8bit",
             report_to="none", remove_unused_columns=False, dataloader_num_workers=2)
_sig = inspect.signature(TrainingArguments.__init__).parameters
_ok = {k: v for k, v in _want.items() if k in _sig}
_dropped = sorted(set(_want) - set(_ok))
print("transformers", __import__("transformers").__version__)
if _dropped:
    print("  unsupported in this build, dropped:", _dropped)
targs = TrainingArguments(**_ok)

# Trainer picks NotebookProgressCallback in a notebook, which renders an IPython
# widget. Widgets never reach Kaggle's log stream, so a batch run shows no
# progress at all. Force plain text instead.
from transformers import TrainerCallback
class PlainProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            el = time.time() - _t0
            done = state.global_step / max(state.max_steps, 1)
            eta = el / done - el if done > 0 else 0
            print(f"  step {state.global_step}/{state.max_steps} "
                  f"({100*done:.0f}%)  " +
                  "  ".join(f"{k}={v:.4g}" for k, v in logs.items() if isinstance(v, (int, float))) +
                  f"  elapsed {el/60:.0f}m  eta {eta/60:.0f}m", flush=True)

_t0 = time.time()
trainer = Trainer(model=vlm, args=targs, train_dataset=VLMDataset(rows),
                  data_collator=collate, callbacks=[PlainProgress()])
trainer.remove_callback(type(trainer.callback_handler.callbacks[-2]))  # drop the widget one
print(f"training {len(rows)} samples -> {targs.max_steps if targs.max_steps>0 else len(rows)//8} steps")
trainer.train()
vlm.save_pretrained("insightvlm-lora/final")
print("saved -> insightvlm-lora/final")

In [ ]:
vlm.eval()
PROMPT = ("You are a data analyst. Examine this chart and report the statistical "
          "findings it supports. State the dominant pattern, cite the numbers that "
          "justify it, and note any caveat a reader should know.")

def generate(path, max_new_tokens=320):
    msgs = [{"role":"user","content":[{"type":"image","image":path},{"type":"text","text":PROMPT}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msgs)
    inp = processor(text=[text], images=[imgs], return_tensors="pt").to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

CLS = json.loads((DATA/"class_index.json").read_text())["classes"]
DESC = {CLS[c]["desc"]: c for c in CLASSES}
def parse_class(t):
    for d,c in DESC.items():
        if d[:40].lower() in t.lower(): return c
    for c in CLASSES:
        if c.replace("_"," ") in t.lower(): return c
    return None

# CAUTION: the SFT target embeds the class description verbatim, so the model can
# score well by reciting a memorised phrase. The real-vs-synthetic gap is the check.
ev = te.sample(300, random_state=0); out_rows = []
for k,(_,r) in enumerate(ev.iterrows()):
    txt = generate(str(DATA / r.image))
    gt = set(re.findall(r"-?\d+\.\d{2}", str(json.loads(r.statistics_json))))
    pr = set(re.findall(r"-?\d+\.\d{2}", txt))
    out_rows.append({"true":r.label,"pred":parse_class(txt),"provenance":r.provenance,
                     "numeric_fidelity":len(gt&pr)/max(len(pr),1),"text":txt})
    if k % 25 == 0: print(f"\r  {k}/{len(ev)}", end="", flush=True)

v = pd.DataFrame(out_rows); v["ok"] = v.true == v.pred
print(f"\n\nVLM class accuracy   {v.ok.mean():.4f}")
print(f"VLM numeric fidelity {v.numeric_fidelity.mean():.4f}")
display(v.groupby("provenance").ok.agg(["mean","count"]).round(4))
print("\nA large real-vs-synthetic gap means template recitation, not chart reading.")
v.to_csv("nb4_vlm_predictions.csv", index=False)
Path("nb4_results.json").write_text(json.dumps({
    "n_train":len(rows), "n_eval":len(v), "class_accuracy":float(v.ok.mean()),
    "numeric_fidelity":float(v.numeric_fidelity.mean()),
    "by_provenance":v.groupby("provenance").ok.mean().round(4).to_dict()}, indent=2))
print("wrote nb4_results.json")